In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from scipy.spatial.distance import cosine
import nltk
from nltk.tokenize import sent_tokenize
import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import json
import re
import pandas as pd

# To load a dataset from Google Drive, please specify the path to your file.
# For example, if you have a CSV file named 'my_dataset.csv' in your Drive,
# the path might look like '/content/drive/MyDrive/my_dataset.csv'.
# If you don't know the exact path, you can browse your Drive in the left sidebar
# (Files icon) and copy the path.

try:
    # Example for loading a JSON file:
    dataset_path = '/content/drive/MyDrive/10701_Project/train-v2.0.json'
    with open(dataset_path, "r") as f:
      data = json.load(f)

    records = []
    for article in data["data"]:
        for p in article["paragraphs"]:
            context = re.sub(r"\s+", " ", p["context"]).strip()
            for qa in p["qas"]:
                if qa.get("is_impossible"):
                    continue  # skip unanswerable
                for ans in qa.get("answers", []):
                    records.append({
                        "title": article["title"],
                        "context": context,
                        "question": re.sub(r"\s+", " ", qa["question"]).strip(),
                        "answer_text": ans["text"].strip(),
                        "answer_start": ans["answer_start"]
                    })

    df = pd.DataFrame(records).drop_duplicates(subset=["question", "context"])
    df.to_csv("cleaned_squad_dev.csv", index=False)
except FileNotFoundError:
    print(f"Error: Dataset not found at {dataset_path}. Please check the path.")
except Exception as e:
    print(f"An error occurred while loading the dataset: {e}")

# Task
Use the SQuAD dataset loaded into the `df` DataFrame to demonstrate a Semantic Retriever by loading a SentenceTransformer model, generating embeddings for questions and potential answer sentences, identifying answer sentences with the minimal cosine distance to question embeddings, and finally presenting the identified answer sentences for a given question.

##Sentence Transformer

In [ ]:
# Load the pre-trained SentenceTransformer model
model = SentenceTransformer('distilbert-base-nli-stsb-mean-tokens')
# model = SentenceTransformer('all-mpnet-base-v1')

print("SentenceTransformer model 'distilbert-base-nli-stsb-mean-tokens' loaded successfully.")

SentenceTransformer model 'distilbert-base-nli-stsb-mean-tokens' loaded successfully.


In [ ]:
def retrieve_answer_sentence(question, context):
    """
    Retrieves the sentence from context that has minimal distance to the question.

    Args:
        question (str): The question to answer
        context (str): The context paragraph containing the answer

    Returns:
        tuple: (best_sentence, distance, sentence_index)
    """
    try:
        nltk.data.find('tokenizers/punkt')
    except LookupError:
        nltk.download('punkt')
    # Split context into sentences
    sentences = sent_tokenize(context)

    if not sentences:
        return None, float('inf'), -1

    # Encode question
    question_embedding = model.encode(question, convert_to_tensor=False)

    # Encode all sentences
    sentence_embeddings = model.encode(sentences, convert_to_tensor=False)

    # Calculate cosine distances between question and each sentence
    distances = []
    for sent_emb in sentence_embeddings:
        # Cosine distance = 1 - cosine similarity
        dist = cosine(question_embedding, sent_emb)
        distances.append(dist)

    # Find sentence with minimal distance
    min_idx = np.argmin(distances)
    best_sentence = sentences[min_idx]
    min_distance = distances[min_idx]

    return best_sentence, min_distance, min_idx

In [ ]:
def evaluate_retriever(df, sample_size=None):
    """
    Evaluates the semantic retriever on the dataset.

    Args:
        df (pd.DataFrame): DataFrame with questions, contexts, and answers
        sample_size (int): If provided, evaluate on a random sample of this size

    Returns:
        pd.DataFrame: Results with retrieved sentences and evaluation metrics
    """
    if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=42)

    results = []

    print(f"Processing {len(df)} question-context pairs...")
    for count, (idx, row) in enumerate(df.iterrows(), 1):
        question = row['question']
        context = row['context']
        answer_text = row['answer_text']

        # Retrieve best sentence
        best_sentence, distance, sent_idx = retrieve_answer_sentence(question, context)

        # Check if answer is in the retrieved sentence
        answer_found = answer_text.lower() in best_sentence.lower() if best_sentence else False

        results.append({
            'question': question,
            'context': context,
            'answer_text': answer_text,
            'retrieved_sentence': best_sentence,
            'distance': distance,
            'sentence_index': sent_idx,
            'answer_found': answer_found
        })

        if count % 100 == 0:
            print(f"Processed {count}/{len(df)} examples...")

    results_df = pd.DataFrame(results)

    # Calculate accuracy
    accuracy = results_df['answer_found'].mean()
    print(f"\nRetrieval Accuracy: {accuracy:.2%}")
    print(f"Average Distance: {results_df['distance'].mean():.4f}")

    return results_df

In [ ]:
import nltk
nltk.download('punkt_tab')
print("NLTK 'punkt_tab' resource downloaded.")

NLTK 'punkt_tab' resource downloaded.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# Evaluate on a sample (use smaller sample for faster testing)
print("\nEvaluating on sample of 500 examples...")
results_df = evaluate_retriever(df, sample_size=500)

# Save results
output_file = "semantic_retrieval_results.csv"
results_df.to_csv(output_file, index=False)
print(f"\nResults saved to {output_file}")

# Show some statistics
print("\n" + "="*80)
print("STATISTICS")
print("="*80)
print(f"Total examples evaluated: {len(results_df)}")
print(f"Examples with answer found: {results_df['answer_found'].sum()}")
print(f"Retrieval accuracy: {results_df['answer_found'].mean():.2%}")
print(f"Average distance: {results_df['distance'].mean():.4f}")
print(f"Median distance: {results_df['distance'].median():.4f}")
print(f"Min distance: {results_df['distance'].min():.4f}")
print(f"Max distance: {results_df['distance'].max():.4f}")


Evaluating on sample of 500 examples...
Processing 500 question-context pairs...
Processed 100/500 examples...
Processed 200/500 examples...
Processed 300/500 examples...
Processed 400/500 examples...
Processed 500/500 examples...

Retrieval Accuracy: 71.20%
Average Distance: 0.4393

Results saved to semantic_retrieval_results.csv

STATISTICS
Total examples evaluated: 500
Examples with answer found: 356
Retrieval accuracy: 71.20%
Average distance: 0.4393
Median distance: 0.4333
Min distance: 0.0914
Max distance: 0.9163


##TF-IDF Retriever

In [ ]:
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_distances
import unicodedata

nltk.download('punkt_tab')
print("NLTK 'punkt_tab' resource downloaded.")

NLTK 'punkt_tab' resource downloaded.


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def normalize_text(text):
    text = text.lower()
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')
    return text


def retrieve_tfidf(question, context, vectorizer):
  try:
      nltk.data.find('tokenizers/punkt')
  except LookupError:
      nltk.download('punkt')

  #split context into sentences
  sentences = sent_tokenize(context)

  # vectorizer = TfidfVectorizer(stop_words='english') #ignore meaningless words - "a", "the", "of"

  corpus = [question] + sentences #form the whole corpus
  tfidf_matrix = vectorizer.fit_transform(corpus) #caculate the tfidf


  question_vec = tfidf_matrix[0]
  sentence_vecs = tfidf_matrix[1:]
  #distance
  distances = cosine_distances(question_vec, sentence_vecs).flatten()

  #find the closest sentence with respect to the question_vector
  min_idx = np.argmin(distances)
  best_sentence = sentences[min_idx]
  min_distance = distances[min_idx]

  return best_sentence, min_distance, min_idx


def evaluate_tfidf(df, sample_size=None, random_state=42):
  if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=random_state)
  results = []

  for count, (idx, row) in enumerate(df.iterrows(), 1):
    Q = normalize_text(row['question'])
    C = normalize_text(row['context'])
    A = normalize_text(row['answer_text'])



    #get the closest sentence by calculating TFIDF vector
    best_sentence, distance, sent_idx = retrieve_tfidf(Q, C, vectorizer=model)
    #check if correct sentence is found
    found = A.lower() in best_sentence.lower() if best_sentence else False

    if count % 100 == 0:
            print(f"Processed {count}/{len(df)} examples...")


    results.append({
        'question': Q,
        'context': C,
        'answer_text': A,
        'retrieved_sentence': best_sentence,
        'distance': distance,
        'sentence_index': sent_idx,
        'answer_found': found
    })


  results_df = pd.DataFrame(results)
  accuracy = results_df['answer_found'].mean()
  print(f"\nRetrieval Accuracy: {accuracy:.2%}")

  return results_df

In [ ]:
model = TfidfVectorizer(stop_words='english')

print("\nEvaluating")
results_df = evaluate_tfidf(df, sample_size=500, random_state=42)

output_file = "TFIDF_retrieval_results.csv"
results_df.to_csv(output_file, index=False)


print(f"Total examples evaluated: {len(results_df)}")
print(f"Examples with answer found: {results_df['answer_found'].sum()}")
print(f"Retrieval accuracy: {results_df['answer_found'].mean():.2%}")
print(f"Average distance: {results_df['distance'].mean():.4f}")
print(f"Median distance: {results_df['distance'].median():.4f}")
print(f"Min distance: {results_df['distance'].min():.4f}")
print(f"Max distance: {results_df['distance'].max():.4f}")


Evaluating
Processed 100/500 examples...
Processed 200/500 examples...
Processed 300/500 examples...
Processed 400/500 examples...
Processed 500/500 examples...

Retrieval Accuracy: 80.00%
Total examples evaluated: 500
Examples with answer found: 400
Retrieval accuracy: 80.00%
Average distance: 0.7067
Median distance: 0.7371
Min distance: 0.1728
Max distance: 1.0000


##BM25

In [ ]:
from rank_bm25 import BM25Okapi
from nltk.tokenize import sent_tokenize, word_tokenize

def retrieve_bm25(question, context):
    #Split context into sentences
    sentences = sent_tokenize(context)

    #Tokenize sentences
    tokenized_corpus = [word_tokenize(s.lower()) for s in sentences]

    #Build BM25 index
    bm25 = BM25Okapi(tokenized_corpus)

    #Tokenize question
    query_tokens = word_tokenize(question.lower())

    #Compute BM25 scores
    scores = bm25.get_scores(query_tokens)

    #Find the best matching sentence
    best_idx = int(np.argmax(scores))
    best_sentence = sentences[best_idx]
    best_score = scores[best_idx]

    return best_sentence, -best_score, best_idx

def evaluate_bm25(df, sample_size=None, random_state=42):
    if sample_size:
        df = df.sample(n=min(sample_size, len(df)), random_state=random_state)

    results = []

    for count, (idx, row) in enumerate(df.iterrows(), 1):
        Q = normalize_text(row['question'])
        C = normalize_text(row['context'])
        A = normalize_text(row['answer_text'])

        best_sentence, distance, sent_idx = retrieve_bm25(Q, C)
        found = A.lower() in best_sentence.lower() if best_sentence else False

        results.append({
            'question': Q,
            'context': C,
            'answer_text': A,
            'retrieved_sentence': best_sentence,
            'distance': distance,
            'sentence_index': sent_idx,
            'answer_found': found
        })

        if count % 100 == 0:
            print(f"Processed {count}/{len(df)} examples...")

    results_df = pd.DataFrame(results)
    accuracy = results_df['answer_found'].mean()
    print(f"\nBM25 Retrieval Accuracy: {accuracy:.2%}")

    return results_df



In [ ]:
print("\nEvaluating BM25 retriever on 500 examples...")
results_bm25 = evaluate_bm25(df, sample_size=500)

print(f"Total examples evaluated: {len(results_bm25)}")
print(f"Examples with answer found: {results_bm25['answer_found'].sum()}")
print(f"Retrieval accuracy: {results_bm25['answer_found'].mean():.2%}")
print(f"Average distance: {results_bm25['distance'].mean():.4f}")
print(f"Median distance: {results_bm25['distance'].median():.4f}")
print(f"Min distance: {results_bm25['distance'].min():.4f}")
print(f"Max distance: {results_bm25['distance'].max():.4f}")

results_bm25.to_csv("bm25_retrieval_results.csv", index=False)
print("Results saved to bm25_retrieval_results.csv")



Evaluating BM25 retriever on 500 examples...
Processed 100/500 examples...
Processed 200/500 examples...
Processed 300/500 examples...
Processed 400/500 examples...
Processed 500/500 examples...

BM25 Retrieval Accuracy: 77.80%
Total examples evaluated: 500
Examples with answer found: 389
Retrieval accuracy: 77.80%
Average distance: -3.0425
Median distance: -2.6248
Min distance: -15.1925
Max distance: 6.4740
Results saved to bm25_retrieval_results.csv
